# NB25 — ablation tables with the corrected A2 row for P3 x spatial (CPU, no data)

Each NB24 A2 run is paired with the reference re-run on the same rebuilt cache; Holm is redone within the cell's ablation family. The forest plot is redrawn from the confirmation-fold table.

In [ ]:
REPO = 'https://github.com/Devguru-codes/Major_project_IIITN.git'
BRANCH = 'feat/implementation'
WORK = '/kaggle/working'
SRC = '/tmp/eegrep-src'   # code checkout stays out of the saved output

import subprocess, sys, os, json, platform, shutil

def sh(cmd, log=None):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-20000:], r.stderr[-5000:], sep='\n', flush=True)
    if log:
        open(log, 'w').write(r.stdout + '\n' + r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')
    return r.stdout

shutil.rmtree(SRC, ignore_errors=True)
sh(f'git clone --depth 1 -b {BRANCH} {REPO} {SRC}')
SHA = sh(f'git -C {SRC} rev-parse --short HEAD').strip()
sh(f'pip install -q -e "{SRC}[dev,stats]"', log=f'{WORK}/pip_install.log')
PY = f'cd {SRC} && {sys.executable} -m eegrep'
print('git sha', SHA)

In [ ]:
sh(f'cd {SRC} && {sys.executable} -m pytest -q -p no:cacheprovider tests/test_audit.py', log=f'{WORK}/pytest_nb25.txt')

In [ ]:
sh(f'{PY} ablate-replace --table {SRC}/reports/nb07/tables/ablations.csv --ref {SRC}/reports/a2_spatial/ref_selection.jsonl --abl {SRC}/reports/a2_spatial/a2_selection.jsonl --note "NB24 rebuilt cache, own reference" --out {WORK}/ablations_selection.csv', log=f'{WORK}/replace_selection.log')

In [ ]:
sh(f'{PY} ablate-replace --table {SRC}/reports/nb07/tables/confirm/ablations.csv --ref {SRC}/reports/a2_spatial/ref_confirmation.jsonl --abl {SRC}/reports/a2_spatial/a2_confirmation.jsonl --note "NB24 rebuilt cache, own reference" --out {WORK}/ablations_confirmation.csv --figure-dir {WORK}/figures', log=f'{WORK}/replace_confirmation.log')

In [ ]:
import torch
sh(f'{sys.executable} -m pip freeze', log=f'{WORK}/pip_freeze.txt')
env = {'git_sha': SHA, 'python': platform.python_version(), 'torch': torch.__version__,
       'cuda': torch.cuda.is_available(), 'cpu_count': os.cpu_count(),
       'kaggle_image': os.environ.get('KAGGLE_DOCKER_IMAGE', 'unknown')}
json.dump(env, open(f'{WORK}/environment.json', 'w'), indent=1)
print(env)